# 09 — Layers and the forward pass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/009-layers-and-forward-pass.ipynb)

A small network is just a few neurons stacked, and the forward pass is the arithmetic that turns an input into an output. Here you compute a 2-2-1 network by hand, draw what each hidden unit does to the input plane, and then remove the bend to see the whole thing collapse.

## 1. The numbers

Two inputs feed two hidden units, and those two units feed one output. Each arrow carries a weight, each unit carries a bias, and the hidden units bend their scores with ReLU before passing them on.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

x = np.array([1.0, -1.0])                             # the input: two features
W1 = np.array([[1.0, 0.5], [-1.0, 2.0]])              # layer 1: row per hidden unit, column per input
b1 = np.array([0.0, -0.5])                            # layer 1: one bias per hidden unit
W2 = np.array([[1.0, -2.0]])                          # layer 2: one output unit, two weights
b2 = 0.25                                             # layer 2: a single bias

z1 = W1 @ x + b1                                      # hidden scores, before any bend
a1 = np.maximum(0.0, z1)                              # hidden activations, after ReLU
z2 = (W2 @ a1)[0] + b2                                # the network's single output
print("z1 =", z1)                                     # [0.5, -3.5]
print("a1 =", a1)                                     # [0.5, 0.0]: the second unit is clipped
print("z2 =", z2)                                     # 0.75

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `x = ...` — the input vector, two features.
- `W1 = ...` — the first weight matrix; each row belongs to one hidden unit.
- `b1 = ...` — one bias per hidden unit.
- `W2 = ...` — the second weight matrix; it produces the single output.
- `b2 = 0.25` — the output bias.
- `z1 = W1 @ x + b1` — matrix times vector, then the bias: two hidden scores.
- `a1 = ...` — ReLU applied elementwise to both hidden scores.
- `z2 = (W2 @ a1)[0] + b2` — dot the two activations with the output weights.
- `print("z1 = ...")` — the hidden scores before the bend.
- `print("a1 = ...")` — the hidden activations after the bend.
- `print("z2 = ...")` — the network's single output.

## 2. Step by step

The matrix product hides two separate dot products. Written out, each hidden unit takes both inputs, weights them, adds its bias, and then the output unit does the same to the two activations.

In [ ]:
def forward(x, W1, b1, W2, b2):                       # the whole 2-2-1 network in one call
    z1 = W1 @ x + b1                                  # hidden scores: two numbers
    a1 = np.maximum(0.0, z1)                          # hidden activations: the bend
    z2 = (W2 @ a1)[0] + b2                            # output: one number
    return z1, a1, z2                                 # hand back every stage

print("step 1 - hidden raw scores z1:")               # label the stage
for k in range(2):                                    # one line per hidden unit
    print(f"  unit {k+1}: {W1[k,0]:+.2f}*{x[0]:+.2f} + {W1[k,1]:+.2f}*{x[1]:+.2f} + {b1[k]:+.2f} = {z1[k]:+.2f}")   # expand the dot product
print("step 2 - after ReLU, a1 =", np.maximum(0.0, z1))   # negatives become zero
print("step 3 - output z2 =", (W2 @ np.maximum(0.0, z1))[0] + b2)   # second dot product
print("step 4 - throughput of the output unit:")      # label the last stage
for k in range(2):                                    # one term per hidden unit
    print(f"  {W2[0,k]:+.2f} * {a1[k]:+.2f}")        # each unit's contribution to z2
print("check - forward() agrees:", forward(x, W1, b1, W2, b2)[2])   # 0.75, same as above

**Line by line**

- `def forward(x, W1, b1, W2, b2)` — the full pass, reusable.
- `z1 = W1 @ x + b1` — the hidden layer's scores.
- `a1 = np.maximum(0.0, z1)` — the ReLU bend.
- `z2 = (W2 @ a1)[0] + b2` — the output layer; `[0]` unwraps the length-1 array.
- `return z1, a1, z2` — every stage, so any of them can be inspected.
- `print("step 1 ...")` — labels the first stage.
- `for k in range(2)` — walk the two hidden units.
- `print(f"  unit {k+1}: ...")` — spell out the two products and the bias.
- `print("step 2 ...")` — the two activations after clipping.
- `print("step 3 ...")` — the same output recomputed in one expression.
- `print("step 4 ...")` — labels the output unit's terms.
- `for k in range(2)` — walk the two contributions again.
- `print(f"  {W2[0,k]:+.2f} * ...")` — each hidden unit's share of the output.
- `forward(x, ...)[2]` — index 2 picks the output from the returned triple.

## 3. Plot: what each hidden unit sees

Run the hidden layer over a grid of inputs and draw each unit's activation as a colour map. One unit is on for some inputs and off for others, and the boundary between those two regions is a straight line set by its weights.

In [ ]:
import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab

grid = np.linspace(-2, 2, 41)                         # 41 input values per axis
heat = np.zeros((2, len(grid), len(grid)))            # two units, one plane each
for i, x1 in enumerate(grid):                         # walk the first input
    for j, x2 in enumerate(grid):                     # walk the second input
        zz = W1 @ np.array([x1, x2]) + b1             # forward pass into the hidden layer
        heat[:, i, j] = np.maximum(0.0, zz)           # store both ReLU'd values

fig, axes = plt.subplots(1, 2, figsize=(8, 3))        # two panels, one per hidden unit
for k in range(2):                                    # draw each unit's map
    im = axes[k].imshow(heat[k], origin="lower", extent=[-2, 2, -2, 2], cmap="viridis")   # the activation plane
    axes[k].set_title(f"hidden unit {k+1}")           # which unit this is
    axes[k].set_xlabel("x1")                          # x-axis label
    axes[k].set_ylabel("x2")                          # y-axis label
    fig.colorbar(im, ax=axes[k])                      # the colour scale for this panel
plt.suptitle("each hidden unit splits the plane along its own line")   # the takeaway
plt.show()                                            # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — Colab already ships the plotting library.
- `grid = np.linspace(-2, 2, 41)` — the input values used on both axes.
- `heat = np.zeros((2, 41, 41))` — an empty box for two activation maps.
- `for i, x1 in enumerate(grid)` — the outer loop over the first input.
- `for j, x2 in enumerate(grid)` — the inner loop over the second input.
- `zz = W1 @ np.array([x1, x2]) + b1` — one forward pass for this grid point.
- `heat[:, i, j] = ...` — write both hidden values into their maps.
- `plt.subplots(1, 2, ...)` — two panels side by side.
- `for k in range(2)` — one pass per hidden unit.
- `axes[k].imshow(...)` — the map, with `origin="lower"` so up is up.
- `axes[k].set_title(...)` — says which unit the panel belongs to.
- `axes[k].set_xlabel("x1")` — names the first input axis.
- `axes[k].set_ylabel("x2")` — names the second input axis.
- `fig.colorbar(...)` — the colour scale, so the brightness has units.
- `plt.suptitle(...)` — the takeaway, above both panels.
- `plt.show()` — draws the figure.

## 4. Interactive: move the hidden bias

The bias sets where the bend starts. Drag it and the two hidden scores slide up or down together; the readout reports the output, which changes only while a unit is above zero.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(b1_value=0.0):                               # b1_value comes from the slider
    b1_i = np.array([b1_value, b1_value])             # both units get the same shift
    z1_i = W1 @ x + b1_i                              # hidden scores at this bias
    a1_i = np.maximum(0.0, z1_i)                      # hidden activations
    z2_i = (W2 @ a1_i)[0] + b2                        # the output
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.bar([0, 1], z1_i, width=0.35, color="tab:grey", label="z1 before ReLU")   # raw scores
    plt.bar([0.35, 1.35], a1_i, width=0.35, color="tab:blue", label="a1 after ReLU")   # activations
    plt.axhline(0, color="grey", linewidth=0.6)       # the zero line
    plt.xticks([0.175, 1.175], ["unit 1", "unit 2"])  # label each pair of bars
    plt.ylim(-5, 3)                                   # fixed axis, so nothing jumps
    plt.xlabel("hidden unit")                         # x-axis label
    plt.ylabel("value")                               # y-axis label
    plt.title(f"b1 = {b1_value:+.2f} -> z2 = {z2_i:+.2f}")   # live readout
    plt.legend()                                      # name the bars
    plt.show()                                        # draw it

interact(show, b1_value=(-2.0, 2.0, 0.5))             # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(b1_value=0.0)` — the function that redraws one frame.
- `b1_i = ...` — the slider value, copied into both hidden biases.
- `z1_i = W1 @ x + b1_i` — the hidden scores at this bias.
- `a1_i = ...` — the same scores after ReLU.
- `z2_i = ...` — the output, recomputed so the title can report it.
- `plt.figure(...)` — a new canvas for each frame.
- `plt.bar([0, 1], z1_i, ...)` — the raw scores, in grey.
- `plt.bar([0.35, 1.35], a1_i, ...)` — the activations, in blue.
- `plt.axhline(0, ...)` — the zero line.
- `plt.xticks(...)` — names the units under their bars.
- `plt.ylim(-5, 3)` — a fixed range, so the bars do not jump.
- `plt.xlabel(...)` — names the hidden units.
- `plt.ylabel(...)` — names the measured value.
- `plt.title(f"...")` — the bias and the resulting output, live.
- `plt.legend()` — matches each bar to its label.
- `plt.show()` — draws the frame.
- `interact(show, b1_value=(...))` — the slider, its range and its step.

## 5. When it breaks: no bend means one layer

Remove the ReLU and the two layers multiply out into a single weight matrix, so the network can only draw straight lines. Below, the bent network and its flattened copy are plotted on the same axes while one input sweeps from -3 to 3.

In [ ]:
W_combined = W2 @ W1                                  # two linear layers fused into one
print("W2 @ W1 =", W_combined)                        # a single 1x2 row: the hidden width vanished

path = np.linspace(-3, 3, 121)                        # sweep one input, hold the other at 0
inp = np.stack([path, np.zeros_like(path)])           # shape (2, 121): one column per test point
z_lin = (W2 @ np.maximum(0.0, W1 @ inp + b1[:, None]))[0] + b2   # the real net, with the bend
z_flat = (W_combined @ inp)[0] + (W2 @ b1)[0] + b2    # the collapsed net: one straight line

plt.figure(figsize=(6, 3))                            # new figure
plt.plot(path, z_lin, color="tab:blue", linewidth=2, label="2 layers, with ReLU")   # bends at a corner
plt.plot(path, z_flat, color="tab:red", linestyle="--", label="W2 @ W1, no ReLU")   # one straight line
plt.xlabel("x1 (x2 fixed at 0)")                      # x-axis label
plt.ylabel("network output z2")                       # y-axis label
plt.title("without the bend, two layers are just one layer")   # the failure
plt.legend()                                          # name the curves
plt.show()                                            # render figure

**Line by line**

- `W_combined = W2 @ W1` — matrix multiply fuses the two linear maps.
- `print("W2 @ W1 = ...")` — the fused map has one row and two columns.
- `path = ...` — 121 values to sweep the first input.
- `inp = np.stack(...)` — a 2-by-121 batch, with the second input pinned at zero.
- `z_lin = ...` — the real network's output: hidden scores, the bend, then layer two.
- `z_flat = ...` — the fused network; `(W2 @ b1)[0] + b2` folds the biases in too.
- `plt.figure(...)` — a new canvas.
- `plt.plot(path, z_lin, ...)` — the bent curve, in blue.
- `plt.plot(path, z_flat, ...)` — the straight line, dashed red.
- `plt.xlabel(...)` — says which input was swept.
- `plt.ylabel(...)` — names the output.
- `plt.title(...)` — states why the collapse matters.
- `plt.legend()` — matches each curve to its label.
- `plt.show()` — draws the figure.

## Your turn

In section 4, drag `b1` until hidden unit 2 leaves the flat part of ReLU, and note the output before and after. Then change `W2` in section 1 to `[[1.0, 0.0]]` and predict how `z2` depends on the second hidden unit.